In [118]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

### MAP信息汇总

In [119]:
Inefficient_standard_map = {
'吸油烟机':6000,
'灶具':6000,
'烤箱':1500,
'蒸箱':1500,
'微波炉':1500,
'蒸烤烹饪机':1500,
'蒸烤微烹饪机':1500,
'蒸微':1500,
'灶消烹饪机':1500,
'灶蒸烹饪机':1500,
'灶蒸烤烹饪机':1500,
'消毒柜':1000,
'热水器':900,
'两用炉':900,
'家用净水机':400,
'商用净水机':400,
'水槽洗碗机':2400,
'嵌入式洗碗机':2400,
}

产品组_产品类别_map = {
    '吸油烟机':'吸油烟机',
    '灶具':'灶具',
    '烤箱':'蒸烤微',
    '蒸箱':'蒸烤微',
    '微波炉':'蒸烤微',
    '蒸烤烹饪机':'蒸烤微',
    '蒸烤微烹饪机':'蒸烤微',
    '蒸微':'蒸烤微',
    '灶消烹饪机':'灶集成',
    '灶蒸烹饪机':'灶集成',
    '灶蒸烤烹饪机':'灶集成',
    '消毒柜':'消毒柜',
    '热水器':'热水器',
    '两用炉':'热水器',
    '家用净水机':'净水机',
    '商用净水机':'净水机',
    '水槽洗碗机':'洗碗机',
    '嵌入式洗碗机':'洗碗机',
}

### 读取单型号贡献报告中整理出的中间数据（处理了物流、财务、渠道、产品组、国内、核算价）

In [120]:
df = pd.read_excel(r'C:\Users\zhangbon\Desktop\清洗了物流-财务-渠道-产品组-国内.xlsx')
df['商品编码'] = df['商品编码'].astype(str)
df.head()
print(len(df))

306178


### 将产品的生命周期相关信息匹配进来

In [121]:
df_product_life = pd.read_excel(r"C:\Users\zhangbon\Desktop\报告\单型号贡献\产品生命周期状态全表20250703.xlsx")
# 转换物料号列为字符串类型，并只取前13位
df_product_life[['物料号']] = df_product_life[['物料号']].astype(str).applymap(lambda x: x[:13])
df_product_life = df_product_life[df_product_life['物料号'].str.len()>10]
df_product_life = df_product_life.drop_duplicates('物料号')
df_product_life_map_df = df_product_life[['物料号','产品状态','产品型号']]

C:\Users\zhangbon\AppData\Local\Temp\ipykernel_8380\2065005041.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_product_life[['物料号']] = df_product_life[['物料号']].astype(str).applymap(lambda x: x[:13])


In [122]:
df1 = pd.merge(df,df_product_life_map_df,how='left',left_on='商品编码',right_on='物料号')
df1

,商品编码,渠道,实际出库数量,产品组,系统核算价,核算价,标准型号,国内/海外,物料号,产品状态,产品型号
0,1009001100002,工程,1,蒸烤烹饪机,3550,3550,ZK50-01-F1.i,国内,1009001100002,退市预警,ZK50-01-F1.i
1,1001002100022,工程,2,吸油烟机,1508,3016,JC03A,国内,1001002100022,量产,CXW-258-JC03A(不带罩)
2,1002003400032,工程,2,灶具,900,1800,TH3B,国内,1002003400032,量产,JZT-TH33B-12T
3,1001002000018,工程,1,吸油烟机,3668,3668,03-X1A,国内,1001002000018,量产,CXW-358-03-X1A
4,1003000500029,工程,1,消毒柜,1550,1550,ZTD100J-J31,国内,1003000500029,量产,ZTD100J-J51E
...,...,...,...,...,...,...,...,...,...,...,...
306173,1013000600009,非零售工程电商,86,商用净水机,7980,686280,FZN400-02-A5,国内,1013000600009,量产,FZN400-02-A5
306174,1013000100033,零售,40,家用净水机,2280,91200,YCZ-JT1800-01-M2E,国内,1013000100033,量产,YCZ-JT1800-01-M2E
306175,1013000100033,非零售工程电商,1,家用净水机,2280,2280,YCZ-JT1800-01-M2E,国内,1013000100033,量产,YCZ-JT1800-01-M2E
306176,1013000100034,零售,44,家用净水机,2280,100320,YCZ-JT1800-01-M2E,国内,1013000100034,量产,YCZ-JT1600-01-M3G


### 选出标准型号下面的型号存在量产阶段的标准型号,并只留下这一部分标准型号的数据（此时不涉及生命周期的筛选）

In [123]:

mass_standard = list(df1[df1['产品状态'] == '量产']['标准型号'].drop_duplicates())
df_calu = df1[df1['标准型号'].isin(mass_standard)]
df_calu = df_calu.reset_index(drop=True)
df_calu


,商品编码,渠道,实际出库数量,产品组,系统核算价,核算价,标准型号,国内/海外,物料号,产品状态,产品型号
0,1001002100022,工程,2,吸油烟机,1508,3016,JC03A,国内,1001002100022,量产,CXW-258-JC03A(不带罩)
1,1002003400032,工程,2,灶具,900,1800,TH3B,国内,1002003400032,量产,JZT-TH33B-12T
2,1001002000018,工程,1,吸油烟机,3668,3668,03-X1A,国内,1001002000018,量产,CXW-358-03-X1A
3,1003000500029,工程,1,消毒柜,1550,1550,ZTD100J-J31,国内,1003000500029,量产,ZTD100J-J51E
4,1018000500033,工程,1,嵌入式洗碗机,3350,3350,JBCD7E-02-V6,国内,1018000500033,退市预警,JBCD7E-02-V6
...,...,...,...,...,...,...,...,...,...,...,...
283435,1013000600009,非零售工程电商,86,商用净水机,7980,686280,FZN400-02-A5,国内,1013000600009,量产,FZN400-02-A5
283436,1013000100033,零售,40,家用净水机,2280,91200,YCZ-JT1800-01-M2E,国内,1013000100033,量产,YCZ-JT1800-01-M2E
283437,1013000100033,非零售工程电商,1,家用净水机,2280,2280,YCZ-JT1800-01-M2E,国内,1013000100033,量产,YCZ-JT1800-01-M2E
283438,1013000100034,零售,44,家用净水机,2280,100320,YCZ-JT1800-01-M2E,国内,1013000100034,量产,YCZ-JT1600-01-M3G


### 计算出每个标准型号的总发货数，在依据其的产品组标准判断，各个各个标准型号是否是低效整机

In [124]:
df_calu['标准型号总发货数'] = df_calu.groupby('标准型号')['实际出库数量'].transform('sum')
df_calu['标准型号是否低效'] = ''
for index, row in df_calu.iterrows():
    if row['标准型号总发货数'] < Inefficient_standard_map[row['产品组']]:
        df_calu.loc[index,'标准型号是否低效'] = '是'
    else:
        df_calu.loc[index,'标准型号是否低效'] = '否'
df_calu

,商品编码,渠道,实际出库数量,产品组,系统核算价,核算价,标准型号,国内/海外,物料号,产品状态,产品型号,标准型号总发货数,标准型号是否低效
0,1001002100022,工程,2,吸油烟机,1508,3016,JC03A,国内,1001002100022,量产,CXW-258-JC03A(不带罩),14752,否
1,1002003400032,工程,2,灶具,900,1800,TH3B,国内,1002003400032,量产,JZT-TH33B-12T,357974,否
2,1001002000018,工程,1,吸油烟机,3668,3668,03-X1A,国内,1001002000018,量产,CXW-358-03-X1A,55760,否
3,1003000500029,工程,1,消毒柜,1550,1550,ZTD100J-J31,国内,1003000500029,量产,ZTD100J-J51E,66461,否
4,1018000500033,工程,1,嵌入式洗碗机,3350,3350,JBCD7E-02-V6,国内,1018000500033,退市预警,JBCD7E-02-V6,8129,否
...,...,...,...,...,...,...,...,...,...,...,...,...,...
283435,1013000600009,非零售工程电商,86,商用净水机,7980,686280,FZN400-02-A5,国内,1013000600009,量产,FZN400-02-A5,221,是
283436,1013000100033,零售,40,家用净水机,2280,91200,YCZ-JT1800-01-M2E,国内,1013000100033,量产,YCZ-JT1800-01-M2E,1258,否
283437,1013000100033,非零售工程电商,1,家用净水机,2280,2280,YCZ-JT1800-01-M2E,国内,1013000100033,量产,YCZ-JT1800-01-M2E,1258,否
283438,1013000100034,零售,44,家用净水机,2280,100320,YCZ-JT1800-01-M2E,国内,1013000100034,量产,YCZ-JT1600-01-M3G,1258,否


### 只保留渠道、产品组、标准型号、标准型号是否低效的数据，并去重。并计算各个产品组的低效标准型号数和总标准型号数

In [125]:
df_calu2 = df_calu[['渠道','产品组','标准型号','标准型号是否低效']].drop_duplicates()
df_calu2 = df_calu2.reset_index(drop=True)
df_calu2

,渠道,产品组,标准型号,标准型号是否低效
0,工程,吸油烟机,JC03A,否
1,工程,灶具,TH3B,否
2,工程,吸油烟机,03-X1A,否
3,工程,消毒柜,ZTD100J-J31,否
4,工程,嵌入式洗碗机,JBCD7E-02-V6,否
...,...,...,...,...
1030,工程,嵌入式洗碗机,JBCD15E-W3,是
1031,非零售工程电商,吸油烟机,EM18C,否
1032,非零售工程电商,吸油烟机,02-EMD20T,是
1033,零售,灶具,TH83B.S,是


In [126]:
df_calu2['产品类别'] = df_calu2['产品组'].map(产品组_产品类别_map)
df_calu2['低效标准型号数'] = df_calu2.groupby('产品类别')['标准型号是否低效'].transform(lambda x: (x=='是').sum())
df_calu2['总标准型号数'] = df_calu2.groupby('产品类别')['标准型号'].transform('nunique')
df_calu2['低效标准型号占比'] = df_calu2['低效标准型号数']/df_calu2['总标准型号数']
df_calu2

,渠道,产品组,标准型号,标准型号是否低效,产品类别,低效标准型号数,总标准型号数,低效标准型号占比
0,工程,吸油烟机,JC03A,否,吸油烟机,69,110,0.627273
1,工程,灶具,TH3B,否,灶具,60,68,0.882353
2,工程,吸油烟机,03-X1A,否,吸油烟机,69,110,0.627273
3,工程,消毒柜,ZTD100J-J31,否,消毒柜,9,22,0.409091
4,工程,嵌入式洗碗机,JBCD7E-02-V6,否,洗碗机,64,73,0.876712
...,...,...,...,...,...,...,...,...
1030,工程,嵌入式洗碗机,JBCD15E-W3,是,洗碗机,64,73,0.876712
1031,非零售工程电商,吸油烟机,EM18C,否,吸油烟机,69,110,0.627273
1032,非零售工程电商,吸油烟机,02-EMD20T,是,吸油烟机,69,110,0.627273
1033,零售,灶具,TH83B.S,是,灶具,60,68,0.882353


In [127]:
df_calu2 = df_calu2[['产品类别','低效标准型号数','低效标准型号占比','总标准型号数']].drop_duplicates()
df_calu2 = df_calu2.reset_index(drop=True)
df_calu2

,产品类别,低效标准型号数,低效标准型号占比,总标准型号数
0,吸油烟机,69,0.627273,110
1,灶具,60,0.882353,68
2,消毒柜,9,0.409091,22
3,洗碗机,64,0.876712,73
4,蒸烤微,33,0.846154,39
5,灶集成,15,0.681818,22
6,热水器,46,0.884615,52
7,净水机,8,0.470588,17


### 统计各个渠道的型号数

In [ ]:
df_calu3 = df_calu[['产品型号','渠道','产品组','标准型号是否低效']].drop_duplicates()
df_calu3['产品类别'] = df_calu3['产品组'].map(产品组_产品类别_map)
df_calu3_retail = df_calu3[df_calu3['渠道']=='零售']
df_calu3_engine = df_calu3[df_calu3['渠道']=='工程']
df_calu3_ecom = df_calu3[df_calu3['渠道']=='电商']
df_calu3_channel_all = df_calu3.copy()


#### 全渠道的分析

In [129]:
df_calu3_channel_all['全渠道低效型号数'] = df_calu3_channel_all.groupby('产品类别')['标准型号是否低效'].transform(lambda x:(x=='是').sum())
df_calu3_channel_all['全渠道型号数'] = df_calu3_channel_all.groupby('产品类别')['产品型号'].transform('nunique')
df_calu3_channel_all['全渠道低效型号占比'] = df_calu3_channel_all['全渠道低效型号数']/df_calu3_channel_all['全渠道型号数']
df_calu3_channel_all = df_calu3_channel_all[['产品类别','全渠道低效型号数','全渠道低效型号占比','全渠道型号数']].drop_duplicates()
df_calu3_channel_all


,产品类别,全渠道低效型号数,全渠道低效型号占比,全渠道型号数
0,吸油烟机,75,0.443787,169
1,灶具,106,0.377224,281
3,消毒柜,9,0.200000,45
4,洗碗机,72,0.847059,85
7,蒸烤微,35,0.760870,46
63,灶集成,24,0.413793,58
87,热水器,50,0.609756,82
531,净水机,8,0.307692,26


#### 零售渠道的统计

In [130]:
df_calu3_retail['零售渠道低效型号数'] = df_calu3_retail.groupby('产品类别')['标准型号是否低效'].transform(lambda x:(x=='是').sum())
df_calu3_retail['零售渠道型号数'] = df_calu3_retail.groupby('产品类别')['产品型号'].transform('nunique')
df_calu3_retail['零售渠道低效型号占比'] = df_calu3_retail['零售渠道低效型号数']/df_calu3_retail['零售渠道型号数']
df_calu3_retail = df_calu3_retail[['产品类别','零售渠道低效型号数','零售渠道低效型号占比','零售渠道型号数']].drop_duplicates()
df_calu3_retail


C:\Users\zhangbon\AppData\Local\Temp\ipykernel_8380\3371310840.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_calu3_retail['零售渠道低效型号数'] = df_calu3_retail.groupby('产品类别')['标准型号是否低效'].transform(lambda x:(x=='是').sum())
C:\Users\zhangbon\AppData\Local\Temp\ipykernel_8380\3371310840.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_calu3_retail['零售渠道型号数'] = df_calu3_retail.groupby('产品类别')['产品型号'].transform('nunique')
C:\Users\zhangbon\AppData\Local\Temp\ipykernel_8380\3371310840.py:3: SettingWithCo

,产品类别,零售渠道低效型号数,零售渠道低效型号占比,零售渠道型号数
525,吸油烟机,21,0.256098,82
528,灶具,40,0.220994,181
529,消毒柜,4,0.121212,33
531,净水机,2,0.100000,20
532,热水器,21,0.318182,66
536,洗碗机,17,0.340000,50
547,灶集成,15,0.272727,55
562,蒸烤微,7,0.280000,25


#### 工程渠道的统计

In [131]:
df_calu3_engine['工程渠道低效型号数'] = df_calu3_engine.groupby('产品类别')['标准型号是否低效'].transform(lambda x:(x=='是').sum())
df_calu3_engine['工程渠道型号数'] = df_calu3_engine.groupby('产品类别')['产品型号'].transform('nunique')
df_calu3_engine['工程渠道低效型号占比'] = df_calu3_engine['工程渠道低效型号数']/df_calu3_engine['工程渠道型号数']
df_calu3_engine = df_calu3_engine[['产品类别','工程渠道低效型号数','工程渠道低效型号占比','工程渠道型号数']].drop_duplicates()
df_calu3_engine


C:\Users\zhangbon\AppData\Local\Temp\ipykernel_8380\3390502439.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_calu3_engine['工程渠道低效型号数'] = df_calu3_engine.groupby('产品类别')['标准型号是否低效'].transform(lambda x:(x=='是').sum())
C:\Users\zhangbon\AppData\Local\Temp\ipykernel_8380\3390502439.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_calu3_engine['工程渠道型号数'] = df_calu3_engine.groupby('产品类别')['产品型号'].transform('nunique')
C:\Users\zhangbon\AppData\Local\Temp\ipykernel_8380\3390502439.py:3: SettingWithCo

,产品类别,工程渠道低效型号数,工程渠道低效型号占比,工程渠道型号数
0,吸油烟机,17,0.236111,72
1,灶具,6,0.086957,69
3,消毒柜,2,0.086957,23
4,洗碗机,9,0.264706,34
7,蒸烤微,5,0.227273,22
63,灶集成,4,0.285714,14
87,热水器,2,0.086957,23
34194,净水机,1,0.090909,11


#### 电商渠道的统计

In [132]:
df_calu3_ecom['电商渠道低效型号数'] = df_calu3_ecom.groupby('产品类别')['标准型号是否低效'].transform(lambda x:(x=='是').sum())
df_calu3_ecom['电商渠道型号数'] = df_calu3_ecom.groupby('产品类别')['产品型号'].transform('nunique')
df_calu3_ecom['电商渠道低效型号占比'] = df_calu3_ecom['电商渠道低效型号数']/df_calu3_ecom['电商渠道型号数']
df_calu3_ecom = df_calu3_ecom[['产品类别','电商渠道低效型号数','电商渠道低效型号占比','电商渠道型号数']].drop_duplicates()
df_calu3_ecom


C:\Users\zhangbon\AppData\Local\Temp\ipykernel_8380\2874558597.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_calu3_ecom['电商渠道低效型号数'] = df_calu3_ecom.groupby('产品类别')['标准型号是否低效'].transform(lambda x:(x=='是').sum())
C:\Users\zhangbon\AppData\Local\Temp\ipykernel_8380\2874558597.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_calu3_ecom['电商渠道型号数'] = df_calu3_ecom.groupby('产品类别')['产品型号'].transform('nunique')
C:\Users\zhangbon\AppData\Local\Temp\ipykernel_8380\2874558597.py:3: SettingWithCopyWarnin

,产品类别,电商渠道低效型号数,电商渠道低效型号占比,电商渠道型号数
32500,吸油烟机,13,0.160494,81
32506,灶具,28,0.195804,143
32512,蒸烤微,12,0.292683,41
32526,消毒柜,0,0.000000,9
32571,热水器,13,0.295455,44
32669,净水机,1,0.058824,17
32885,灶集成,0,0.000000,15
33710,洗碗机,16,0.432432,37


### 把三个渠道的数据合并起来，并输出标准型号的统计分析和渠道型号的统计分析

In [133]:
df_qudao = pd.DataFrame()
df_qudao['产品类别'] = ['吸油烟机','灶具','蒸烤微','灶集成','消毒柜','热水器','净水机','洗碗机']
df_qudao = pd.merge(df_qudao,df_calu3_retail,on='产品类别',how='left')
df_qudao = pd.merge(df_qudao,df_calu3_engine,on='产品类别',how='left')
df_qudao = pd.merge(df_qudao,df_calu3_ecom,on='产品类别',how='left')
df_qudao = pd.merge(df_qudao,df_calu3_channel_all,on='产品类别',how='left')
df_qudao
df_stand_temp = pd.DataFrame()
df_stand_temp['产品类别'] = ['吸油烟机','灶具','蒸烤微','灶集成','消毒柜','热水器','净水机','洗碗机']
df_calu2 = pd.merge(df_stand_temp,df_calu2,on='产品类别',how='left')
#输出df_qudao和df_calu2，写到一个excel里面
with pd.ExcelWriter('C:\\Users\\zhangbon\\Desktop\\低效.xlsx') as writer:
    df_qudao.to_excel(writer, sheet_name='各渠道统计',index=False)
    df_calu2.to_excel(writer, sheet_name='标准型号统计',index=False)

